In [3]:
import os
import cv2
import torch
import pandas as pd
from tqdm import tqdm

# ===== CONFIG =====
images_path = "D:/hoctap/HK6/DACN1/images"
species_csv = "species_list.csv"
output_csv = "megadetector_results.csv"

conf_threshold = 0.2  # MegaDetector thường dùng thấp

# ===== LOAD SPECIES =====
species_df = pd.read_csv(species_csv)
species_list = species_df["scientific_name"].tolist()

# ===== LOAD MODEL =====
model_path = "md_v5a.0.0.pt"  # tải từ MegaDetector release
model = torch.hub.load('ultralytics/yolov5', 'custom', path=model_path)

# class mapping của MegaDetector
label_map = {
    0: "animal",
    1: "person",
    2: "vehicle"
}

results = []

error_count = 0
animal_count = 0
no_animal_count = 0

# ===== LOOP =====
for species in species_list:
    species_folder = os.path.join(images_path, species)

    if not os.path.exists(species_folder):
        print(f"⚠️ Không có folder: {species}")
        continue

    for file in tqdm(os.listdir(species_folder), desc=f"{species}"):
        path = os.path.join(species_folder, file)

        if not file.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue

        try:
            img = cv2.imread(path)
            if img is None:
                raise Exception("Corrupt image")

            # predict
            results_md = model(img)

            detections = results_md.xyxy[0]  # x1,y1,x2,y2,conf,class

            found_animal = False
            found_human = False
            found_vehicle = False

            for *box, conf, cls in detections:
                if conf < conf_threshold:
                    continue

                cls = int(cls)

                if cls == 0:
                    found_animal = True
                elif cls == 1:
                    found_human = True
                elif cls == 2:
                    found_vehicle = True

            if found_animal:
                label = "animal"
                animal_count += 1
            elif found_human:
                label = "human"
            elif found_vehicle:
                label = "vehicle"
            else:
                label = "no_animal"
                no_animal_count += 1

        except Exception as e:
            label = "error"
            error_count += 1

        results.append([species, path, label])

# ===== SAVE CSV =====
df = pd.DataFrame(results, columns=["species", "path", "status"])
df.to_csv(output_csv, index=False)

# ===== STATS =====
print("\n===== KẾT QUẢ =====")
print(f"Animal: {animal_count}")
print(f"No animal: {no_animal_count}")
print(f"Error: {error_count}")
print(f"Saved: {output_csv}")

Using cache found in C:\Users\ADMIN/.cache\torch\hub\ultralytics_yolov5_master
YOLOv5  2026-3-21 Python-3.11.5 torch-2.6.0+cpu CPU



Exception: [Errno 2] No such file or directory: 'md_v5a.0.0.pt'. Cache may be out of date, try `force_reload=True` or see https://docs.ultralytics.com/yolov5/tutorials/pytorch_hub_model_loading for help.

In [1]:
import urllib.request

url = "https://storage.googleapis.com/public-datasets-lila/megadetector/md_v5a.0.0.pt"
output_path = "md_v5a.0.0.pt"

urllib.request.urlretrieve(url, output_path)

print("Download xong!")

HTTPError: HTTP Error 404: Not Found